# TDI 101524 — TDRB (TD Re-insurance Barbados) DQ Check

## Purpose
Checks data availability for all source tables and columns used in the TDI 101524 TDRB metric calculations.

## Pattern
Follows the TDW PT 101015 DQ check approach: one check per (table, column), recording which CDEs use each column.

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/GAMLConnections

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/Settings

In [ ]:
from datetime import date

# DQ results table
TABLE_NAME = SNAPSHOT_CATALOGUE_NAME + '.' + TABLE_NAME_DATA_AVA_SEG
today_date = str(date.today())

lob_id = '101524'
lob_desc = 'TDI - TD Re-insurance Barbados'

def insertDQTable(SNAPSHOT_CATALOGUE_NAME, TABLE_NAME_DATA_AVA_SEG,
                  lob_id, cde_no, cde_desc, source, src_table_name,
                  data_element, availability_pct, today_date):
    query = ("INSERT INTO " + SNAPSHOT_CATALOGUE_NAME + "." + TABLE_NAME_DATA_AVA_SEG
             + " VALUES('" + lob_id + "','" + cde_no + "','" + cde_desc + "','"
             + source + "','" + src_table_name + "','" + data_element + "','"
             + str(availability_pct) + "','" + today_date + "')")
    spark.sql(query)
    return True

def check_column(cde_no, cde_desc, source, src_table, column):
    """Check availability of a column and insert into DQ table."""
    try:
        query = f"""
            SELECT count(1) as total,
                   count(CASE WHEN cast({column} as STRING) IS NOT NULL
                         AND trim(cast({column} as STRING)) != '' THEN 1 END) as valid
            FROM {src_table}
        """
        result = spark.sql(query).collect()[0]
        total, valid = result[0], result[1]
        pct = round(100.0 * valid / total, 2) if total > 0 else 0
        insertDQTable(SNAPSHOT_CATALOGUE_NAME, TABLE_NAME_DATA_AVA_SEG,
                      lob_id, cde_no, cde_desc, source, src_table,
                      column, pct, today_date)
        print(f"  {column}: {valid}/{total} = {pct}%")
    except Exception as e:
        print(f"  {column}: ERROR - {str(e)}")

print(f'DQ Check for {lob_id} ({lob_desc})')
print(f'Results table: {TABLE_NAME}')
print(f'Run date: {today_date}')

## Segment Data Sources

In [ ]:
# ============================================================
# Source: ra_adido_2025.101524_tdi_barbados_customer
# Main TDRB customer table
# ============================================================
print('ra_adido_2025.101524_tdi_barbados_customer')
print('=' * 60)

check_column('SD1,SD3,SD1.1,SD1.3,1.6,1.7,4.1A,4.1B,4.1C,6.5B,SD6', 'Segment/Centralized', 'ADIDO',
             'ra_adido_2025.101524_tdi_barbados_customer', 'Customer_Unique_ID')

check_column('SD3,1.6,1.7,ABAC1.1', 'Segment', 'ADIDO',
             'ra_adido_2025.101524_tdi_barbados_customer', 'ISOA_Country_Code')

check_column('SD4', 'Segment', 'ADIDO',
             'ra_adido_2025.101524_tdi_barbados_customer', 'SIC_Code')

check_column('ABAC1.3', 'ABAC', 'ADIDO',
             'ra_adido_2025.101524_tdi_barbados_customer', 'Standardized_SIC_Code')

check_column('SD5', 'Segment', 'ADIDO',
             'ra_adido_2025.101524_tdi_barbados_customer', 'Legal_Entity_Classification')

check_column('SD1,SD1.1,SD1.3,1.6,1.7,4.1A,4.1B,4.1C,6.5B,SD6', 'Segment', 'ADIDO',
             'ra_adido_2025.101524_tdi_barbados_customer', 'Length_of_Relationship')

check_column('SD1,SD1.1,SD1.3,1.6,1.7,6.5B', 'Segment', 'ADIDO',
             'ra_adido_2025.101524_tdi_barbados_customer', 'Customer_Status')

check_column('4.1B,4.1C', 'Segment', 'ADIDO',
             'ra_adido_2025.101524_tdi_barbados_customer', 'DeliveryChannel')

check_column('1.8,1.9,3.17,3.18', 'Centralized', 'ADIDO',
             'ra_adido_2025.101524_tdi_barbados_customer', 'Customer_Name_Beneficial_Owner')

check_column('1.8,1.9', 'Centralized', 'ADIDO',
             'ra_adido_2025.101524_tdi_barbados_customer', 'Counterparty_Name')

## Branch / Office Data

In [ ]:
# ============================================================
# Source: ra_adido_2025.barbados_registered_office
# Branch/office data for 5.1-5.8, ABAC5.1
# ============================================================
print('ra_adido_2025.barbados_registered_office')
print('=' * 60)

check_column('5.1,5.2,5.3,5.4,5.5,5.6,5.7,5.8,ABAC5.1', 'Branch', 'ADIDO',
             'ra_adido_2025.barbados_registered_office', 'address')

check_column('5.2,5.3,5.4,5.5,5.6,5.7,5.8,ABAC5.1', 'Branch', 'ADIDO',
             'ra_adido_2025.barbados_registered_office', 'country')

check_column('5.1,ABAC5.1', 'Branch', 'ADIDO',
             'ra_adido_2025.barbados_registered_office', 'office')

## Transaction Data

In [ ]:
# ============================================================
# Source: ra_adido_2025.barbados_transaction_fy25
# Transaction data for SD7, SD8, 3.1-3.16, ABAC3.1, ABAC3.2
# ============================================================
print('ra_adido_2025.barbados_transaction_fy25')
print('=' * 60)

check_column('SD7,SD8,3.1,3.2,3.3,3.4,3.5,3.6,3.7,3.8,ABAC3.1,ABAC3.2', 'Transaction', 'ADIDO',
             'ra_adido_2025.barbados_transaction_fy25', 'FileName')

check_column('SD7,SD8,3.1,3.2,3.3,3.4,3.5,3.6,3.7,3.8,ABAC3.1,ABAC3.2', 'Transaction', 'ADIDO',
             'ra_adido_2025.barbados_transaction_fy25', 'Jurisdiction')

# Note: Column names with hyphens require backtick quoting in Spark SQL
check_column('SD7,SD8,3.9,3.10,3.11,3.12,3.13,3.14,3.15,3.16,ABAC3.2', 'Transaction', 'ADIDO',
             'ra_adido_2025.barbados_transaction_fy25', '`Incoming-Deposits`')

check_column('SD7,SD8,3.9,3.10,3.11,3.12,3.13,3.14,3.15,3.16,ABAC3.2', 'Transaction', 'ADIDO',
             'ra_adido_2025.barbados_transaction_fy25', '`Outgoing-Payments`')

## Centralized / Risk Rating Data

In [ ]:
# ============================================================
# Source: ra_adido_2025.barbados_centralized_fy25
# Centralized risk data for 1.1-1.5, SD2, ABAC1.2
# ============================================================
print('ra_adido_2025.barbados_centralized_fy25')
print('=' * 60)

check_column('1.1,1.2,1.3,1.4,1.5,SD2,ABAC1.2', 'Centralized', 'ADIDO',
             'ra_adido_2025.barbados_centralized_fy25', 'Customer_Unique_ID')

check_column('1.1,1.2,1.3,1.4,1.5', 'Centralized', 'ADIDO',
             'ra_adido_2025.barbados_centralized_fy25', 'TDRISKSRATING')

# Note: Column name with special chars requires backtick quoting
check_column('SD2,ABAC1.2', 'Centralized', 'ADIDO',
             'ra_adido_2025.barbados_centralized_fy25', '`POLITICALLYEXPOSEDPERSON/HDIassociates`')

## Sanctions / STR Data

In [ ]:
# ============================================================
# Source: ra_adido_2025.true_sanction_match_2025
# True sanctions matching for 1.8
# ============================================================
print('ra_adido_2025.true_sanction_match_2025')
print('=' * 60)

check_column('1.8', 'Centralized', 'ADIDO',
             'ra_adido_2025.true_sanction_match_2025', 'Customername')

check_column('1.8', 'Centralized', 'ADIDO',
             'ra_adido_2025.true_sanction_match_2025', 'CustomerType')

In [ ]:
# ============================================================
# Source: ra_adido_2025.customer_sanctioned_blocked_property_2025
# Blocked property for 1.9
# ============================================================
print('ra_adido_2025.customer_sanctioned_blocked_property_2025')
print('=' * 60)

check_column('1.9', 'Centralized', 'ADIDO',
             'ra_adido_2025.customer_sanctioned_blocked_property_2025', 'CUSTOMERIDIMPACTED')

In [ ]:
# ============================================================
# Source: ra_adido_2025.barbados_str
# STR data for 3.17, 3.18
# ============================================================
print('ra_adido_2025.barbados_str')
print('=' * 60)

check_column('3.17,3.18', 'Centralized', 'ADIDO',
             'ra_adido_2025.barbados_str', 'details')

check_column('3.17,3.18', 'Centralized', 'ADIDO',
             'ra_adido_2025.barbados_str', 'date')

check_column('3.17,3.18', 'Centralized', 'ADIDO',
             'ra_adido_2025.barbados_str', 'amount')

## Reference / Lookup Tables

In [ ]:
# ============================================================
# Source: ra_adido_2025.industry_ref_list_ca2025
# Industry risk ratings for SD4
# ============================================================
print('ra_adido_2025.industry_ref_list_ca2025')
print('=' * 60)

check_column('SD4', 'Reference', 'ADIDO',
             'ra_adido_2025.industry_ref_list_ca2025', 'Industry_Code')

check_column('SD4', 'Reference', 'ADIDO',
             'ra_adido_2025.industry_ref_list_ca2025', 'Updated_Risk_Rating')

In [ ]:
# ============================================================
# Source: ra_adido_2025.entity_ref_list_ca2025
# Entity type risk ratings for SD5
# ============================================================
print('ra_adido_2025.entity_ref_list_ca2025')
print('=' * 60)

check_column('SD5', 'Reference', 'ADIDO',
             'ra_adido_2025.entity_ref_list_ca2025', 'Entity_Type_Name')

check_column('SD5', 'Reference', 'ADIDO',
             'ra_adido_2025.entity_ref_list_ca2025', 'Updated_Risk_Ratings')

In [ ]:
# ============================================================
# Source: ra_adido_2025.country_ref_list_ca2025
# Country risk for 5.2-5.5, 3.1-3.4, 3.9-3.12, ABAC1.1
# ============================================================
print('ra_adido_2025.country_ref_list_ca2025')
print('=' * 60)

check_column('5.2,5.3,5.4,5.5,3.1,3.2,3.3,3.4,3.9,3.10,3.11,3.12', 'Reference', 'ADIDO',
             'ra_adido_2025.country_ref_list_ca2025', 'Country_Name')

check_column('5.2,5.3,5.4,5.5,3.1,3.2,3.3,3.4,3.9,3.10,3.11,3.12', 'Reference', 'ADIDO',
             'ra_adido_2025.country_ref_list_ca2025', '`2025_Risk_Rating`')

check_column('ABAC1.1', 'Reference', 'ADIDO',
             'ra_adido_2025.country_ref_list_ca2025', 'ISOA_Country_Code')

In [ ]:
# ============================================================
# Source: ra_fy25_view.sanctions_country_risk_rating_fy2025
# Sanctions risk for 1.6, 5.5-5.8, 3.5-3.8, 3.13-3.16
# ============================================================
print('ra_fy25_view.sanctions_country_risk_rating_fy2025')
print('=' * 60)

check_column('1.6,5.5,5.6,5.7,5.8,3.5,3.6,3.7,3.8,3.13,3.14,3.15,3.16', 'Reference', 'ADIDO',
             'ra_fy25_view.sanctions_country_risk_rating_fy2025', 'Country_Name')

check_column('1.6,5.5,5.6,5.7,5.8,3.5,3.6,3.7,3.8,3.13,3.14,3.15,3.16', 'Reference', 'ADIDO',
             'ra_fy25_view.sanctions_country_risk_rating_fy2025', 'RiskRating')

## ABAC Data Sources

In [ ]:
# ============================================================
# Source: ra_adido_2025.TD_Country_Risk_Rating_ABAC
# ABAC country risk for ABAC1.1, ABAC3.1, ABAC3.2, ABAC5.1
# ============================================================
print('ra_adido_2025.TD_Country_Risk_Rating_ABAC')
print('=' * 60)

check_column('ABAC1.1,ABAC5.1', 'ABAC', 'ADIDO',
             'ra_adido_2025.TD_Country_Risk_Rating_ABAC', 'FinalABACRating')

check_column('ABAC3.1,ABAC3.2,ABAC5.1', 'ABAC', 'ADIDO',
             'ra_adido_2025.TD_Country_Risk_Rating_ABAC', 'Jurisdiction')

check_column('ABAC1.1', 'ABAC', 'ADIDO',
             'ra_adido_2025.TD_Country_Risk_Rating_ABAC', '`Alpha-3Code`')

In [ ]:
# ============================================================
# Source: ra_adido_2025.Industry_Reference_List_08222025_df
# ABAC industry classification for ABAC1.3
# ============================================================
print('ra_adido_2025.Industry_Reference_List_08222025_df')
print('=' * 60)

check_column('ABAC1.3', 'ABAC', 'ADIDO',
             'ra_adido_2025.Industry_Reference_List_08222025_df', 'IndustryCode')

check_column('ABAC1.3', 'ABAC', 'ADIDO',
             'ra_adido_2025.Industry_Reference_List_08222025_df', 'ABACRse')

In [ ]:
# ============================================================
# Source: ra_adhor_data.101524_cde_results_fy2024
# Historical FY2024 results for 6.5A
# ============================================================
print('ra_adhor_data.101524_cde_results_fy2024')
print('=' * 60)

check_column('6.5A', 'Segment', 'ADIDO',
             'ra_adhor_data.101524_cde_results_fy2024', 'values')

check_column('6.5A', 'Segment', 'ADIDO',
             'ra_adhor_data.101524_cde_results_fy2024', 'CDE')

## Summary

In [ ]:
# ============================================================
# Display all DQ results for this AU
# ============================================================
results = spark.sql(f"""
    SELECT * FROM {TABLE_NAME}
    WHERE lob_id = '{lob_id}'
      AND run_date = '{today_date}'
    ORDER BY src_table_name, data_element
""")

print(f'Total DQ checks: {results.count()}')
display(results)